# 06 — DistilBERT: SHAP-Guided Training-Example Refinement

**Research project:** Explainability-Guided Feature Analysis for Reducing False Positives in Phishing Email Detection Without Increasing False Negatives  
**Model:** DistilBERT  
**Experiment:** Context-based selection and weighted fine-tuning of original training examples

This notebook presents an experiment that used SHAP explanations and contextual similarity to identify useful training examples. The purpose was to reduce false-positive predictions while maintaining the baseline level of phishing detection. The experiment is presented independently of the Logistic Regression and Random Forest studies.

## 1. Research objective

The official DistilBERT baseline produced **31 false positives (FP)** and **22 false negatives (FN)** on a test set of **10,716 emails**. The refinement was considered successful only if it reduced FP below 31 while keeping FN at or below 22.

**Table 6.1. Research acceptance criteria**

| Measure | Official baseline | Required outcome |
|:---|---:|:---|
| False positives (FP) | 31 | Fewer than 31 |
| False negatives (FN) | 22 | No more than 22 |
| Test emails | 10,716 | Same test set |

The dataset uses **0 = legitimate** and **1 = phishing**. Throughout this project, **TP** means a legitimate email correctly classified as legitimate; **FP** means a legitimate email classified as phishing; **FN** means a phishing email classified as legitimate; and **TN** means a phishing email correctly classified as phishing.

## 2. Dataset and model foundation

The experiment started from the permanently saved official DistilBERT baseline checkpoint. The original training split was retained, and the refinement did not add test emails to the training data.

**Table 6.2. Dataset used in the experiment**

| Dataset component | Email count |
|:---|---:|
| Cleaned dataset | 53,577 |
| Original training split | 42,861 |
| Original test split | 10,716 |

The refinement used the original training emails with additional weighting for selected examples. This allowed the experiment to emphasize informative cases without replacing the original training distribution.

## 3. SHAP-guided selection of training examples

The analysis began with the **31 legitimate emails incorrectly predicted as phishing** by the official baseline. Each false-positive email was compared with three contextually similar, correctly classified phishing emails. These comparisons helped examine how the model handled similar content in the two email classes.

SHAP explanations supported the comparison by identifying tokens that influenced the phishing predictions. The analysis then searched the **original training split** for legitimate and phishing examples that matched the contextual and explanation patterns. Selected examples were checked against their training labels and duplicate selections were removed.

**Table 6.3. Training-example selection procedure**

| Stage | Method | Purpose |
|:---|:---|:---|
| Error review | Examine 31 baseline false positives | Identify legitimate emails affected by incorrect phishing predictions |
| Contextual comparison | Match three correctly classified phishing emails per false positive | Compare related content across the two classes |
| SHAP interpretation | Compare token-level explanations | Identify evidence associated with the predictions |
| Training-data retrieval | Search the original 42,861 training emails | Find related legitimate and phishing examples |
| Selection checks | Confirm labels and remove duplicates | Prepare reliable examples for weighted fine-tuning |

The contextual comparison generated **93 matching positions** (31 × 3). Some emails appeared in more than one comparison. The recorded selection therefore contained **90 unique** correctly classified phishing emails, rather than 93 distinct emails.

## 4. Controlled weighted fine-tuning

The saved baseline model was fine-tuned for one additional epoch. All **42,861 original training emails** remained available. Ordinary training examples had a weight of 1.0, while selected examples received greater emphasis according to their role in the refinement.

**Table 6.4. Recorded fine-tuning configuration**

| Setting | Value |
|:---|:---|
| Starting checkpoint | Saved official DistilBERT baseline |
| Original training emails | 42,861 |
| Fine-tuning epochs | 1 |
| Learning rate | 2 × 10⁻⁶ (0.000002) |
| Physical batch size | 2 |
| Gradient accumulation steps | 4 |
| Effective batch size | 8 |
| Weight decay | 0.01 |
| Maximum gradient norm | 1.0 |
| Warm-up ratio | 0.05 |
| Standard training-example weight | 1.0 |
| Selected difficult legitimate-example weight | 4.0 |
| Maximum selected phishing-example weight | 1.50 |
| Classification threshold | 0.50 (unchanged) |

The additional weight for selected legitimate examples encouraged the model to learn from cases resembling the baseline false positives. A smaller, controlled weight was used for selected phishing examples to support phishing detection. The baseline checkpoint was preserved separately.

## 5. Evaluation of the refined model

The saved refined model was evaluated on the original **10,716-email test split**, using the unchanged decision threshold of 0.50.

**Table 6.5. Classification outcomes on the same test set (10,716 emails)**

| Classification outcome | Official baseline | Weighted refinement | Change |
|:---|---:|---:|---:|
| TP — correctly classified legitimate | 4,717 | 4,719 | +2 |
| FP — legitimate classified as phishing | **31** | **29** | **−2** |
| FN — phishing classified as legitimate | **22** | **24** | **+2** |
| TN — correctly classified phishing | 5,946 | 5,944 | −2 |
| **Total test emails** | **10,716** | **10,716** | **0** |

The refined model produced two fewer false positives than the baseline (31 to 29). At the same time, false negatives increased by two (22 to 24). These figures describe the net changes in classification outcomes; they do not establish that the same individual emails changed prediction.

## 6. Performance metrics

Accuracy, phishing-class precision, recall, and F1-score can be calculated from the recorded classification counts. Because this project uses TP and TN labels differently from the usual binary-classification notation, the calculations below use the **phishing-class counts** directly.

**Table 6.6. Performance comparison (six decimal places)**

| Metric | Official baseline | Weighted refinement |
|:---|---:|---:|
| Accuracy | 0.995054 | 0.995054 |
| Precision (phishing) | 0.994813 | 0.995145 |
| Recall (phishing) | 0.996314 | 0.995979 |
| F1-score (phishing) | 0.995563 | 0.995562 |
| False positives | **31** | **29** |
| False negatives | **22** | **24** |

The following code independently checks the confusion-matrix totals and calculates the classification metrics. It does not retrain the model.

In [ ]:
import pandas as pd

counts = {
    "Official baseline": {"TP": 4717, "FP": 31, "FN": 22, "TN": 5946},
    "Weighted refinement": {"TP": 4719, "FP": 29, "FN": 24, "TN": 5944},
}

rows = []
for experiment, c in counts.items():
    total = sum(c.values())
    assert total == 10716, f"Unexpected test-set size: {experiment}"
    phishing_precision = c["TN"] / (c["TN"] + c["FP"])
    phishing_recall = c["TN"] / (c["TN"] + c["FN"])
    phishing_f1 = 2 * phishing_precision * phishing_recall / (phishing_precision + phishing_recall)
    rows.append({
        "Experiment": experiment,
        "Accuracy": (c["TP"] + c["TN"]) / total,
        "Precision": phishing_precision,
        "Recall": phishing_recall,
        "F1-score": phishing_f1,
        "FP": c["FP"],
        "FN": c["FN"],
    })

metrics = pd.DataFrame(rows).set_index("Experiment")
print(metrics.to_string(formatters={
    metric: "{:.6f}".format for metric in ["Accuracy", "Precision", "Recall", "F1-score"]
}))

assert counts["Weighted refinement"]["FP"] == 29
assert counts["Weighted refinement"]["FN"] == 24


## 7. Interpretation and conclusion

The SHAP-guided training-example refinement reduced false positives from **31 to 29**, following the additional emphasis placed on selected training examples. However, false negatives increased from **22 to 24**.

The experiment therefore **did not satisfy the combined research objective**. It demonstrated that reducing false positives alone was insufficient; the intervention also needed to preserve the baseline's ability to detect phishing emails. The result supported further investigation of targeted methods that could reduce false positives while keeping false negatives at or below the baseline.

**Outcome:** False positives decreased by two, but false negatives increased by two. The refinement was not selected as the final model because it did not meet both acceptance criteria.